In [32]:
from dotenv import load_dotenv
import os
import shutil

from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
import gradio as gr

In [33]:
openai = OpenAI()
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [19]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [20]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model=MODEL)

In [21]:
retriever.invoke("Who is Avery")

[Document(id='8996f8cb-1300-4195-a584-002c6060306f', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the

In [24]:
llm.invoke("who is avery")

AIMessage(content="Avery is a given name that can be used for both males and females. It can also be a surname. Without additional context, it's difficult to determine which specific Avery you're referring to. If you can provide more details—such as a full name, profession, or context—I’d be happy to help with more specific information!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 11, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c6c2d9662', 'id': 'chatcmpl-DWkB1fKiBU8tw7rCtVUtOaFNvl8PX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dab5b-249b-7713-a4aa-2c0bdbf08d83-0', tool_calls=[], invalid_tool_calls=[

In [25]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company insureLLM,
You are chatting with a user about InsureLLM,
If relevent , uset the given context to answer any question,
If you don't know the answer, say so.
Context:
{context}
"""

In [27]:
def answer_question(question: str, history):
  docs = retriever.invoke(question)
  context = "\n\n".join(doc.page_content for doc in docs)
  system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
  response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
  return response.content


In [30]:
answer_question("who is Avery?", [])

APIConnectionError: Connection error.

In [34]:
from bs4 import BeautifulSoup
import requests

In [38]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]




headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [ ]:

context = """You are a system that triages a list of links and their content form a web scraping tool 
and provide the links that are most relevent and have content"""


user_prompt = "find the relevent links of this URL"


article = """
You are an expreienced journalist, you take the links found on the page {link}
You are also creative and funny, so you take the news and put a spin on it to be funny

"""




def relevent_links(link):
  context = """You are a system that triages a list of links and their content form a web scraping tool 
  and provide the links that are most relevent and have content"""





  article = """
  You are an expreienced journalist, you take the links found on the page {link}
  You are also creative and funny, so you take the news and put a spin on it to be funny

  """




  def relevent_links(link):
    system_prompt = context + "\n\n".join(fetch_website_links(link))
    print(system_prompt)
    response = openai.chat.completions.create( 
      model=MODEL,
      messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}

      ]
    )

    rel_liks = response.choices[0].message.content
    print(rel_liks)
   
   

In [44]:
relevent_links("https://apnews.com/sports")